# 🚀 Real-ESRGAN Batch Image Upscaler (Google Colab GPU)

Aplikasi batch upscale gambar dengan perbesaran **2x** atau **4x** menggunakan GPU T4.  
- **Format Output**: JPG Kualitas Tinggi (`quality=96`, subsampling `4:4:4` - ukuran file tidak drop).
- **Anti-OOM**: Tiling 512x512 aman untuk foto resolusi tinggi.
- **Interaktif**: Dilengkapi progress bar visual, notifikasi selesai, dan tombol reset batch.

---  
**Pastikan GPU T4 Aktif**: Klik menu `Runtime` -> `Change runtime type` -> pilih `T4 GPU` -> `Save`.

In [ ]:
# 1. Periksa GPU (Harus muncul Tesla T4)
!nvidia-smi

In [ ]:
# 2. Siapkan Model & Folder Kerja
import os
!mkdir -p weights inputs outputs

print("Memeriksa file model RealESRGAN_x4plus.pth...")
!wget -c -q --show-progress "https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth" -P weights/
print("Model siap digunakan!")

In [ ]:
# 3. Siapkan Engine Upscaler (Pure PyTorch)
%%writefile upscale_batch.py
import argparse, glob, math, os, shutil, sys
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F

class ResidualDenseBlock(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.conv1 = nn.Conv2d(num_feat, num_grow_ch, 3, 1, 1)
        self.conv2 = nn.Conv2d(num_feat + num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv3 = nn.Conv2d(num_feat + 2 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv4 = nn.Conv2d(num_feat + 3 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.conv5 = nn.Conv2d(num_feat + 4 * num_grow_ch, num_grow_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat((x, x1), 1)))
        x3 = self.lrelu(self.conv3(torch.cat((x, x1, x2), 1)))
        x4 = self.lrelu(self.conv4(torch.cat((x, x1, x2, x3), 1)))
        x5 = self.conv5(torch.cat((x, x1, x2, x3, x4), 1))
        return x5 * 0.2 + x

class RRDB(nn.Module):
    def __init__(self, num_feat=64, num_grow_ch=32):
        super().__init__()
        self.rdb1 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb2 = ResidualDenseBlock(num_feat, num_grow_ch)
        self.rdb3 = ResidualDenseBlock(num_feat, num_grow_ch)
    def forward(self, x):
        out = self.rdb1(x)
        out = self.rdb2(out)
        out = self.rdb3(out)
        return out * 0.2 + x

class RRDBNet(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32):
        super().__init__()
        self.scale = scale
        self.conv_first = nn.Conv2d(num_in_ch, num_feat, 3, 1, 1)
        self.body = nn.Sequential(*[RRDB(num_feat, num_grow_ch) for _ in range(num_block)])
        self.conv_body = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up1 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_up2 = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_hr = nn.Conv2d(num_feat, num_feat, 3, 1, 1)
        self.conv_last = nn.Conv2d(num_feat, num_out_ch, 3, 1, 1)
        self.lrelu = nn.LeakyReLU(negative_slope=0.2, inplace=True)
    def forward(self, x):
        feat = self.conv_first(x)
        body_feat = self.conv_body(self.body(feat))
        feat = feat + body_feat
        feat = self.lrelu(self.conv_up1(F.interpolate(feat, scale_factor=2, mode="nearest")))
        feat = self.lrelu(self.conv_up2(F.interpolate(feat, scale_factor=2, mode="nearest")))
        out = self.conv_last(self.lrelu(self.conv_hr(feat)))
        return out

def load_model(model_path="weights/RealESRGAN_x4plus.pth", device=None, fp16=True):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"Model tidak ada di: {model_path}")
    model = RRDBNet(num_in_ch=3, num_out_ch=3, scale=4, num_feat=64, num_block=23, num_grow_ch=32)
    try:
        checkpoint = torch.load(model_path, map_location=device, weights_only=False)
    except TypeError:
        checkpoint = torch.load(model_path, map_location=device)
    state_dict = checkpoint.get("params_ema", checkpoint.get("params", checkpoint))
    model.load_state_dict(state_dict, strict=True)
    model.eval().to(device)
    return model.half() if (fp16 and device.type == "cuda") else model

def process_tiles(img_tensor, model, tile=512, tile_pad=10, scale=4):
    batch, channel, height, width = img_tensor.shape
    if tile == 0 or (height <= tile and width <= tile):
        with torch.no_grad():
            return model(img_tensor)
    out_tensor = torch.zeros((batch, channel, height * scale, width * scale), dtype=img_tensor.dtype, device=img_tensor.device)
    tiles_x = math.ceil(width / tile)
    tiles_y = math.ceil(height / tile)
    for y in range(tiles_y):
        for x in range(tiles_x):
            in_x = x * tile
            in_y = y * tile
            in_x_end = min(in_x + tile, width)
            in_y_end = min(in_y + tile, height)
            in_x_pad = max(in_x - tile_pad, 0)
            in_x_end_pad = min(in_x_end + tile_pad, width)
            in_y_pad = max(in_y - tile_pad, 0)
            in_y_end_pad = min(in_y_end + tile_pad, height)
            tile_in = img_tensor[:, :, in_y_pad:in_y_end_pad, in_x_pad:in_x_end_pad]
            with torch.no_grad():
                tile_out = model(tile_in)
            out_x = in_x * scale
            out_x_end = in_x_end * scale
            out_y = in_y * scale
            out_y_end = in_y_end * scale
            t_out_x = (in_x - in_x_pad) * scale
            t_out_x_end = t_out_x + (in_x_end - in_x) * scale
            t_out_y = (in_y - in_y_pad) * scale
            t_out_y_end = t_out_y + (in_y_end - in_y) * scale
            out_tensor[:, :, out_y:out_y_end, out_x:out_x_end] = tile_out[:, :, t_out_y:t_out_y_end, t_out_x:t_out_x_end]
    return out_tensor

def upscale_image(pil_img, model, device, tile=512, fp16=True, outscale=4.0):
    orig_w, orig_h = pil_img.size
    img = pil_img.convert("RGB")
    np_img = np.array(img, dtype=np.float32) / 255.0
    tensor = torch.from_numpy(np_img).permute(2, 0, 1).unsqueeze(0).to(device)
    if fp16 and device.type == "cuda":
        tensor = tensor.half()
    out_tensor = process_tiles(tensor, model, tile=tile, tile_pad=10, scale=4)
    out_tensor = out_tensor.squeeze(0).float().clamp(0.0, 1.0)
    out_np = (out_tensor.permute(1, 2, 0).cpu().numpy() * 255.0).round().astype(np.uint8)
    res_img = Image.fromarray(out_np)
    if outscale != 4.0:
        res_img = res_img.resize((int(round(orig_w * outscale)), int(round(orig_h * outscale))), Image.Resampling.LANCZOS)
    return res_img

def get_image_files(input_dir="inputs"):
    os.makedirs(input_dir, exist_ok=True)
    valid_exts = ("*.png", "*.jpg", "*.jpeg", "*.webp", "*.bmp")
    files = []
    for ext in valid_exts:
        files.extend(glob.glob(os.path.join(input_dir, ext)))
        files.extend(glob.glob(os.path.join(input_dir, ext.upper())))
    if not files and os.path.exists("/content"):
        for ext in valid_exts:
            for f in glob.glob(f"/content/{ext}") + glob.glob(f"/content/{ext.upper()}"):
                dest = os.path.join(input_dir, os.path.basename(f))
                os.rename(f, dest)
                files.append(dest)
    return sorted(list(set(files)))

def clean_folders(input_dir="inputs", output_dir="outputs"):
    for folder in [input_dir, output_dir]:
        if os.path.exists(folder):
            for f in os.listdir(folder):
                p = os.path.join(folder, f)
                if os.path.isfile(p): os.remove(p)
                elif os.path.isdir(p): shutil.rmtree(p)
    if os.path.exists("hasil_upscale.zip"): os.remove("hasil_upscale.zip")


In [ ]:
# 4. APLIKASI GUI TERSTRUKTUR (LANGKAH 1 -> 4)
import os, glob, shutil
from PIL import Image
import torch
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import files
from upscale_batch import load_model, upscale_image, get_image_files, clean_folders

# --- 1. LANGKAH 1: UPLOAD ---
btn_upload = widgets.Button(
    description="📁 Upload Gambar Bahan",
    button_style="info",
    layout=widgets.Layout(width="240px", height="40px"),
    icon="upload"
)
lbl_file_count = widgets.HTML(
    value="<span style='color: #666; font-size: 14px;'>Belum ada file. Klik tombol upload atau drag gambar ke folder <b>inputs/</b>.</span>"
)

def update_file_count_label():
    fl = get_image_files("inputs")
    if fl:
        lbl_file_count.value = f"<span style='background:#e8f0fe; color:#1a73e8; padding:5px 10px; border-radius:6px; font-weight:bold;'>🟢 {len(fl)} gambar siap di folder inputs/</span>"
    else:
        lbl_file_count.value = "<span style='color: #d93025; font-weight:bold;'>⚠️ 0 gambar terdeteksi. Upload gambar bahan terlebih dahulu!</span>"

def on_upload_clicked(b):
    btn_upload.disabled = True
    lbl_file_count.value = "<span style='color:#1a73e8;'>Membuka dialog upload file...</span>"
    up = files.upload()
    for fn in up.keys():
        os.rename(fn, os.path.join("inputs", fn))
    update_file_count_label()
    btn_upload.disabled = False

btn_upload.on_click(on_upload_clicked)
update_file_count_label()

# --- 2. LANGKAH 2: PENGATURAN ---
scale_choice = widgets.ToggleButtons(
    options=[('2x (Pas & Tajam)', 2.0), ('4x (Maksimal Detail)', 4.0)],
    value=2.0,
    description='Skala:',
    button_style='info',
    layout=widgets.Layout(margin='5px 0')
)

quality_slider = widgets.IntSlider(
    value=96, min=85, max=100, step=1,
    description='Kualitas JPG:',
    layout=widgets.Layout(width='380px')
)

# --- 3. LANGKAH 3: PROSES UPSCALE ---
btn_run = widgets.Button(
    description="🚀 Mulai Upscale Sekarang",
    button_style="primary",
    layout=widgets.Layout(width="260px", height="45px"),
    icon="play"
)

progress_bar = widgets.IntProgress(
    value=0, min=0, max=100,
    bar_style='info',
    orientation='horizontal',
    layout=widgets.Layout(width='100%', height='25px', display='none')
)

lbl_status = widgets.HTML(value="<span style='color: #666;'>Klik tombol 'Mulai Upscale Sekarang' untuk memproses.</span>")

# --- 4. LANGKAH 4: HASIL & UNDUH ---
alert_box = widgets.HTML(value="")

btn_download = widgets.Button(
    description="📦 Unduh Hasil (ZIP)",
    button_style="success",
    layout=widgets.Layout(width="220px", height="40px"),
    icon="download"
)

btn_clean = widgets.Button(
    description="🧹 Reset / Hapus Batch Ini",
    button_style="danger",
    layout=widgets.Layout(width="240px", height="40px"),
    icon="trash"
)

def on_run_clicked(b):
    fl = get_image_files("inputs")
    if not fl:
        alert_box.value = "<div style='padding:10px 14px; background:#fce8e6; color:#c5221f; border-radius:6px; font-weight:bold;'>⚠️ Folder inputs/ kosong! Silakan upload gambar terlebih dahulu di Langkah 1.</div>"
        return

    # Lock tombol saat memproses
    btn_run.disabled = True
    btn_upload.disabled = True
    btn_download.disabled = True
    btn_clean.disabled = True
    alert_box.value = ""
    
    progress_bar.layout.display = 'block'
    progress_bar.max = len(fl)
    progress_bar.value = 0
    progress_bar.bar_style = 'info'
    lbl_status.value = "<b>[0/" + str(len(fl)) + "] Menginisialisasi Model AI ke GPU...</b>"
    
    s = scale_choice.value
    q = quality_slider.value
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = load_model("weights/RealESRGAN_x4plus.pth", device, fp16=True)
    
    os.makedirs("outputs", exist_ok=True)
    success_count = 0
    for idx, img_path in enumerate(fl, 1):
        fn = os.path.basename(img_path)
        lbl_status.value = f"<b>[{idx}/{len(fl)}] Sedang memproses:</b> {fn} ..."
        try:
            with Image.open(img_path) as src:
                out = upscale_image(src, model, device, tile=512, fp16=True, outscale=s)
                out_name = f"{os.path.splitext(fn)[0]}_upscaled_{int(s)}x.jpg"
                out.save(os.path.join("outputs", out_name), format="JPEG", quality=q, subsampling=0, optimize=True)
                w, h = out.size
                success_count += 1
                lbl_status.value = f"<b>[{idx}/{len(fl)}] Selesai:</b> {out_name} ({w}x{h} px)"
        except Exception as e:
            lbl_status.value = f"<b>[{idx}/{len(fl)}] GAGAL:</b> {fn} ({e})"
        progress_bar.value = idx

    progress_bar.bar_style = 'success'
    lbl_status.value = f"<b>Selesai!</b> {success_count}/{len(fl)} gambar berhasil di-upscale."
    alert_box.value = f"<div style='padding:14px 18px; background:#e6f4ea; border:1px solid #ceead6; color:#137333; border-radius:8px; font-size:15px; font-weight:bold; margin-top:10px;'>🎉 <b>SELESAI 100%!</b> Sebanyak {success_count} gambar telah berhasil di-upscale ({int(s)}x). Silakan klik tombol <b>'Unduh Hasil (ZIP)'</b> di bawah ini.</div>"
    
    btn_run.disabled = False
    btn_upload.disabled = False
    btn_download.disabled = False
    btn_clean.disabled = False

def on_download_clicked(b):
    out_files = glob.glob("outputs/*_upscaled_*.jpg")
    if not out_files:
        alert_box.value = "<div style='padding:10px 14px; background:#fce8e6; color:#c5221f; border-radius:6px; font-weight:bold;'>❌ Folder outputs/ kosong! Silakan jalankan Langkah 3 terlebih dahulu.</div>"
        return
    lbl_status.value = "<b>Mengompres file zip...</b>"
    shutil.make_archive("hasil_upscale", "zip", "outputs")
    size_mb = os.path.getsize("hasil_upscale.zip") / (1024 * 1024)
    lbl_status.value = f"<b>Mengunduh hasil_upscale.zip ({size_mb:.2f} MB) ke komputer Anda...</b>"
    files.download("hasil_upscale.zip")

def on_clean_clicked(b):
    clean_folders("inputs", "outputs")
    update_file_count_label()
    progress_bar.layout.display = 'none'
    progress_bar.value = 0
    lbl_status.value = "<span style='color: #666;'>Folder telah di-reset. Siap untuk batch baru.</span>"
    alert_box.value = "<div style='padding:10px 14px; background:#e8f0fe; color:#1a73e8; border-radius:6px; font-weight:bold;'>🧹 Folder inputs/ dan outputs/ telah dikosongkan. Gambar lama tidak akan tercampur di batch selanjutnya!</div>"

btn_run.on_click(on_run_clicked)
btn_download.on_click(on_download_clicked)
btn_clean.on_click(on_clean_clicked)

# --- SUSUNAN TAMPILAN KARTU TERSTRUKTUR ---
card_style = "border: 1px solid #dadce0; border-radius: 8px; padding: 14px 18px; margin-bottom: 12px; background: #ffffff; box-shadow: 0 1px 2px rgba(60,64,67,0.1);"

step1 = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 10px 0; color:#1a73e8;'>📁 Langkah 1: Unggah Gambar Bahan</h4>"),
    widgets.HBox([btn_upload, lbl_file_count], layout=widgets.Layout(align_items='center'))
], layout=widgets.Layout(extra_style=card_style))

step2 = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 10px 0; color:#1a73e8;'>⚙️ Langkah 2: Pengaturan Kualitas</h4>"),
    scale_choice,
    quality_slider
], layout=widgets.Layout(extra_style=card_style))

step3 = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 10px 0; color:#1a73e8;'>🚀 Langkah 3: Eksekusi & Indikator Progres</h4>"),
    btn_run,
    widgets.Box([progress_bar], layout=widgets.Layout(margin='10px 0 6px 0')),
    lbl_status
], layout=widgets.Layout(extra_style=card_style))

step4 = widgets.VBox([
    widgets.HTML("<h4 style='margin:0 0 10px 0; color:#1a73e8;'>📦 Langkah 4: Hasil Unduhan & Reset Batch</h4>"),
    alert_box,
    widgets.HBox([btn_download, btn_clean], layout=widgets.Layout(margin='8px 0 0 0'))
], layout=widgets.Layout(extra_style=card_style))

dashboard = widgets.VBox([
    widgets.HTML("<h2 style='color:#202124; margin-bottom:4px;'>🖼️ Panel Kontrol Image Upscaler</h2><p style='color:#5f6368; margin-top:0;'>Ikuti langkah 1 sampai 4 secara berurutan:</p>"),
    step1, step2, step3, step4
])

display(dashboard)